# Titik perkiraan kecamatan untuk visualisasi tugas kuliah

Jalankan **Run All** dari folder proyek/Archives dengan kernel `.venv` (dependensi `shapely`).
Memakai master dan poligon lokal proyek, tanpa layanan geocoding atau password.
Target: fasilitas baseline tanpa koordinat, dikurangi keputusan diterima yang sudah tercatat lokal.
Ini bukan pembacaan langsung status database terbaru.

Titik representatif berada di dalam kecamatan, **bukan lokasi bangunan puskesmas**.
Beberapa fasilitas dapat bertumpuk di titik yang sama. Semua hasil berstatus perkiraan,
tidak terverifikasi, dan **tidak boleh digunakan untuk analisis cakupan radius aktual**.
Koordinat sumber/master tidak ditimpa; ekspor dipisahkan dari data analisis.
WebGIS menerapkan pendekatan serupa saat menampilkan fasilitas dengan lokasi kosong.


In [1]:
import json, hashlib
from pathlib import Path
from collections import Counter
from shapely.geometry import shape, mapping

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/processed/facility-master.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka dari checkout proyek yang memiliki data/processed.')
hashes = {}
def read(relative):
    raw = (ROOT / relative).read_bytes()
    hashes[str(relative)] = hashlib.sha256(raw).hexdigest()
    return json.loads(raw.decode('utf-8-sig'))
master = read('data/processed/facility-master.json')['facilities']
boundaries = {f['properties']['region_code']: shape(f['geometry'])
              for f in read('data/processed/boundaries/analysis.geojson')['features']}
accepted = set()
for path in (ROOT / 'docs/pekerjaan/WF-068').glob('*live-review.json'):
    decision = read(path.relative_to(ROOT))
    if decision.get('decision') == 'accepted' and decision.get('facility_code'):
        accepted.add(str(decision['facility_code']))
features = []
for facility in master:
    if facility['baseline_coordinate_present'] or facility['facility_code'] in accepted:
        continue
    boundary = boundaries[facility['region_code']]
    if boundary.is_empty or not boundary.is_valid:
        raise ValueError('Geometri kecamatan tidak valid: ' + facility['region_code'])
    point = boundary.representative_point()
    assert boundary.covers(point)
    features.append({'type': 'Feature', 'id': facility['facility_code'], 'geometry': mapping(point),
                     'properties': {'facility_code': facility['facility_code'],
                                    'name': facility['attributes']['nama_puskesmas'],
                                    'district_code': facility['region_code'],
                                    'district_name': facility['attributes']['kecamatan'],
                                    'regency_name': facility['attributes']['kabupaten'],
                                    'location_kind': 'district_estimate', 'coordinate_status': 'missing',
                                    'precision_level': 'district', 'location_verified': False,
                                    'coverage_eligible': False,
                                    'note': 'Lokasi perkiraan berdasarkan kecamatan; bukan lokasi bangunan'}})
assert len(features) == len({f['id'] for f in features})
assert all(not f['properties']['location_verified'] and not f['properties']['coverage_eligible'] for f in features)
out = ROOT / 'data/exports/district-estimates'
out.mkdir(parents=True, exist_ok=True)
result = {'type': 'FeatureCollection', 'features': features}
(out / 'puskesmas_perkiraan_kecamatan.geojson').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
summary = {'estimated_facilities': len(features), 'district_points': len({f['properties']['district_code'] for f in features}),
           'new_verified': 0, 'database_modified': False, 'input_sha256': hashes}
(out / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps({k: v for k, v in summary.items() if k != 'input_sha256'}, indent=2))
print('Hasil:', out / 'puskesmas_perkiraan_kecamatan.geojson')


{
  "estimated_facilities": 407,
  "district_points": 335,
  "new_verified": 0,
  "database_modified": false
}
Hasil: D:\laragon\www\GIS-puskesmas\data\exports\district-estimates\puskesmas_perkiraan_kecamatan.geojson
